In [1]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels 
import plotly.io as pio 

# import the custom functions for this project
import cytox.core as ctx

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

pd.set_option('display.max_rows', None)
pd.set_option('display.max_Columns', None)

<div style="font-size: 0.6em">

# Import the cleaned data

</div>

In [2]:
df_30min = pd.read_csv(r'..\data\cleaned_data\df_30min.csv')
df_16h = pd.read_csv(r'..\data\cleaned_data\df_16h.csv')

<div style="font-size: 0.6em">

# Define a threshold

</div>

<div style="font-size: 0.6em">

## 30 min data

</div>

In [3]:
df_30min["Sample"].unique()

array(['DMSO_10%_30min', 'DMSO_30%_30min', 'DMSO_100%_30min',
       'Staurosporine_0.1uM_6h', 'Staurosporine_0.3uM_6h',
       'Staurosporine_1uM_6h', 'Saponin_1uM_6h', 'Saponin_3uM_6h',
       'Saponin_9uM_6h', 'Saponin_1uM_30min', 'Saponin_3uM_30min',
       'Saponin_9uM_30min', 'Verteporfin_0.6uM_30min',
       'Verteporfin_2uM_30min', 'Verteporfin_6uM_30min',
       'Digoxin_0.6uM_30min', 'Digoxin_2uM_30min', 'Digoxin_6uM_30min'],
      dtype=object)

In [4]:
"""
Use the DMSO control for determining the threshold as average of signal in 
each channel + 2 standard devisations
"""
threshold_control_30min = df_30min.loc[
    (df_30min["Sample"] == "DMSO_100%_30min")
]  

casp_thresh_gen_30min = ctx.Categorizer(
    normalized_column="Nuclei - Intensity Nucleus Alexa 488 Mean",
    df_untreated = threshold_control_30min, sd=2
)
_, _, casp_thresh_30min = casp_thresh_gen_30min.threshold_generator()

PI_thresh_gen_30min = ctx.Categorizer(
    normalized_column="Nuclei - Intensity Nucleus Alexa 568 Mean",
    df_untreated = threshold_control_30min, sd=2
)

_, _, PI_thresh_30min = PI_thresh_gen_30min.threshold_generator()

The values for control mean, mean-2SD and mean+2SD, respectively:  903.6842772795218, -8311.604669365695, 10118.97322392474
The values for control mean, mean-2SD and mean+2SD, respectively:  774.8943901345291, -8223.045914084627, 9772.834694353684


### 16 h

In [5]:
"""Use the DMSO control for determining the threshold as average of signal in 
each channel + 2 standard devisations"""
threshold_control_16h = df_16h.loc[(df_16h["Sample"] == "DMSO_100%_16h")]  

casp_thresh_gen_16h = ctx.Categorizer(
    normalized_column="Nuclei - Intensity Nucleus Alexa 488 Mean",
    df_untreated = threshold_control_16h, 
    sd=2
)
_, _, casp_thresh_16h = casp_thresh_gen_16h.threshold_generator()

PI_thresh_gen_16h = ctx.Categorizer(
    normalized_column="Nuclei - Intensity Nucleus Alexa 568 Mean",
    df_untreated = threshold_control_16h, 
    sd=2
)

_, _, PI_thresh_16h = PI_thresh_gen_16h.threshold_generator()

The values for control mean, mean-2SD and mean+2SD, respectively:  965.2248549824467, -8370.34334693707, 10300.793056901965
The values for control mean, mean-2SD and mean+2SD, respectively:  628.1948914393736, -5123.921933518551, 6380.3117163972975


<div style="font-size: 0.6em">

# Graphs

</div>

<div style="font-size: 0.6em">

## 30 min 

</div>

In [6]:
# Separate wells: each replicate in its own subplot
fig = ctx.cytotox_raw_plot(
    df_30min,
    title="30 min incubation",
    casp_thresh=casp_thresh_30min,
    pi_thresh=PI_thresh_30min,
    merge_replicates=False       
)

In [11]:
# Combined replicates: all wells of a sample merged into one subplot
fig = ctx.cytotox_raw_plot(
    df_30min,
    title="30 min incubation",
    facet_col_spacing=0.04,
    casp_thresh=casp_thresh_30min,
    pi_thresh=PI_thresh_30min,
    merge_replicates=True,
    fig_show=False
)

# Change the distance of x and y axes labels from axes.
fig.for_each_annotation(
    lambda a: (
        a.update(x=-0.12) if a.text == "PI mean signal intensity"
        else a.update(y=-0.045) 
        if a.text == "Caspase 3/7 average signal intensity"
        else None
    )
)

<div style="font-size: 0.6em">

## 16 h

</div>

In [8]:
# Separate wells: each replicate in its own subplot
fig = ctx.cytotox_raw_plot(
    df_16h,
    facet_row_spacing=0.08,
    title="16 h incubation",
    casp_thresh=casp_thresh_16h,
    pi_thresh=PI_thresh_16h,
    merge_replicates=False,
    fig_show=False     
)

# Change the distance of x axes labels from axes.
fig.for_each_annotation(
    lambda a: (
        a.update(y=-0.12) if a.text == "Caspase 3/7 average signal intensity"
        else None
    )
)

In [9]:
# Combined replicates: all wells of a sample merged into one subplot
fig = ctx.cytotox_raw_plot(
    df_16h,
    title="16 h incubation",
    casp_thresh=casp_thresh_16h,
    pi_thresh=PI_thresh_16h,
    merge_replicates=True,
    fig_show=False
)

# Change the distance of x and y axes labels from axes.
fig.for_each_annotation(
    lambda a: (
        a.update(x=-0.12) if a.text == "PI mean signal intensity"
        else a.update(y=-0.1) 
        if a.text == "Caspase 3/7 average signal intensity"
        else None
    )
)

<div style="font-size: 0.6em">

# Sample overview compiled from single cell data

</div>



<div style="font-size: 0.6em">

## 30 min 

</div>

In [ ]:
cytotox_table_30min = ctx.compute_cytotox_table(
    df_30min,
    casp_thresh=casp_thresh_30min,
    pi_thresh=PI_thresh_30min,
)

cytotox_table_30min.head()

,Sample,Replicate_num,Total_cell_count,live_cell_count,live_cell_percent,Caspase_positive_count,Caspase_positive_percent,PI_positive_count,PI_positive_percent
0,DMSO_10%_30min,1,949,918,96.733404,15,1.580611,26,2.739726
1,DMSO_10%_30min,2,874,850,97.254005,8,0.915332,20,2.288330
2,DMSO_30%_30min,1,924,909,98.376623,8,0.865801,12,1.298701
3,DMSO_30%_30min,2,1268,1257,99.132492,4,0.315457,10,0.788644
4,DMSO_100%_30min,1,1402,1386,98.858773,12,0.855920,11,0.784593


<div style="font-size: 0.6em">

## 16 h 

</div>

In [13]:
cytotox_table_16h = ctx.compute_cytotox_table(
    df_16h,
    casp_thresh=casp_thresh_16h,
    pi_thresh=PI_thresh_16h,
)

cytotox_table_16h.head()

,Sample,Replicate_num,Total_cell_count,live_cell_count,live_cell_percent,Caspase_positive_count,Caspase_positive_percent,PI_positive_count,PI_positive_percent
0,DMSO_10%_16h,1,1,1,100.000000,0,0.000000,0,0.000000
1,DMSO_10%_16h,2,1961,1951,99.490056,5,0.254972,10,0.509944
2,DMSO_30%_16h,1,1899,1892,99.631385,4,0.210637,5,0.263296
3,DMSO_30%_16h,2,1728,1723,99.710648,4,0.231481,4,0.231481
4,DMSO_100%_16h,1,2107,2078,98.623636,23,1.091599,17,0.806834
